# Lab 5 - Workflow Design with LLMs
## Prompt Templates and Output Parsers

**Name:** Jatin Sandip Kolhapure 
**Roll No:** SCFP125014   
**Programme:** MCA  
**Repository:** GenAI_LLM_engineering_SCFP125014

## Objective

This lab builds six practical LLM workflows using LangChain prompt templates and output parsers. Each workflow passes parsed output from one step into the next step, rather than sending the original raw input to every step.

**Model:** `openai/gpt-oss-20b` through the Groq API.

> Run the notebook from top to bottom. The examples use fictional/demo text. Do not add private customer, candidate, or health information.


## 1. Install and import dependencies

Run this cell once in the environment used for the notebook. If you are using a virtual environment, activate it before starting Jupyter.


In [36]:
%pip install -q langchain-core langchain-groq pydantic python-dotenv requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import json
import os
from typing import List

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser


## 2. Configure Groq

Keep the API key in a local `.env` file or environment variable. Never paste the key into a notebook cell or commit the `.env` file to GitHub.


In [38]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. Create a .env file with "
        "GROQ_API_KEY=your_key, or set the environment variable."
    )

MODEL_NAME = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODEL_NAME,
    api_key=GROQ_API_KEY,
    temperature=0.2,
)

print("Groq client is ready.")
print("Model:", MODEL_NAME)


Groq client is ready.
Model: openai/gpt-oss-20b


In [39]:
str_parser = StrOutputParser()

def print_section(title, value):
    print(f"\n{'=' * 12} {title} {'=' * 12}")
    if isinstance(value, BaseModel):
        print(value.model_dump_json(indent=2))
    else:
        print(value)


---
# Exercise 1 - Customer Review to Support Ticket Pipeline

**Input:** `review_text`

**Step 1:** `PromptTemplate` + `PydanticOutputParser` extracts the complaint, product/feature, and sentiment.  
**Step 2:** `ChatPromptTemplate` + `StrOutputParser` writes an internal support ticket using only the parsed fields from Step 1.


In [40]:
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [41]:
class ReviewDetails(BaseModel):
    complaint: str = Field(description="The main issue reported by the customer")
    product_or_feature: str = Field(description="Product or feature mentioned; use 'Not specified' if absent")
    sentiment: str = Field(description="Customer sentiment: Positive, Negative, or Mixed")

review_parser = PydanticOutputParser(pydantic_object=ReviewDetails)

review_extract_prompt = PromptTemplate(
    template=(
        "Extract the key details from the customer review below. "
        "Use only facts in the review. Choose sentiment from Positive, Negative, or Mixed. "
        "If a product or feature is not named, write 'Not specified'.\n\n"
        "Review:\n{review_text}\n\n"
        "{format_instructions}"
    ),
    input_variables=["review_text"],
    partial_variables={"format_instructions": review_parser.get_format_instructions()},
)

ticket_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support ticket writer. Write a concise internal ticket summary based only on the structured details provided. Do not invent account details, dates, or troubleshooting steps."),
    ("human", "Create a support ticket summary from these extracted details:\n{review_details}")
])

review_step_1 = review_extract_prompt | llm | review_parser
review_step_2 = ticket_prompt | llm | str_parser

def customer_review_pipeline(review_text):
    extracted = review_step_1.invoke({"review_text": review_text})
    ticket = review_step_2.invoke({"review_details": extracted.model_dump_json(indent=2)})
    print_section("Step 1 — Structured complaint", extracted)
    print_section("Step 2 — Support ticket summary", ticket)
    return {"structured_complaint": extracted, "support_ticket": ticket}


In [42]:
def print_section(title, value):
    print(f"\n{'=' * 12} {title} {'=' * 12}")

    if isinstance(value, BaseModel):
        print(value.model_dump_json(indent=2))
    else:
        print(value)

In [43]:
review_text = (
    "I've been charged twice for the same monthly plan. The billing page shows two payments, "
    "and I cannot find a way to request a refund. The rest of the app works fine."
)

exercise_1_result = customer_review_pipeline(review_text)



============ Step 1 — Structured complaint ============
{
  "complaint": "Charged twice for the same monthly plan",
  "product_or_feature": "monthly plan",
  "sentiment": "Negative"
}

============ Step 2 — Support ticket summary ============
**Ticket Summary**

- **Issue:** Customer reports being charged twice for the same monthly plan.  
- **Product/Feature:** Monthly plan.  
- **Sentiment:** Negative.


---
# Exercise 2 - Research Abstract to Layperson Summary Pipeline

**Input:** `abstract_text`

**Step 1:** Extract the research question, method, and key finding into a Pydantic model.  
**Step 2:** Explain only that structured extraction in plain language for a non-expert reader.


In [44]:
class ResearchDetails(BaseModel):
    research_question: str = Field(description="The main question the study investigates")
    method: str = Field(description="The method or approach described in the abstract")
    key_finding: str = Field(description="The main finding; preserve uncertainty and limitations")

research_parser = PydanticOutputParser(pydantic_object=ResearchDetails)

research_extract_prompt = PromptTemplate(
    template=(
        "Extract the research question, method, and key finding from this abstract. "
        "Do not add claims that are not in the abstract. If a detail is missing, say 'Not stated'.\n\n"
        "Abstract:\n{abstract_text}\n\n{format_instructions}"
    ),
    input_variables=["abstract_text"],
    partial_variables={"format_instructions": research_parser.get_format_instructions()},
)

layperson_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science communicator explaining research to a general audience. Use clear everyday language, define technical terms briefly, and do not overstate the finding."),
    ("human", "Write a short plain-language summary using only these extracted details:\n{research_details}")
])

research_step_1 = research_extract_prompt | llm | research_parser
research_step_2 = layperson_prompt | llm | str_parser

def research_summary_pipeline(abstract_text):
    extracted = research_step_1.invoke({"abstract_text": abstract_text})
    summary = research_step_2.invoke({"research_details": extracted.model_dump_json(indent=2)})
    print_section("Step 1 — Structured extraction", extracted)
    print_section("Step 2 — Layperson summary", summary)
    return {"extraction": extracted, "summary": summary}


In [45]:
abstract_text = (
    "This study examined whether spaced practice improves vocabulary retention among adult language learners. "
    "Sixty learners were assigned to either daily short review sessions or one longer weekly session over four weeks. "
    "The spaced-practice group recalled more words on a test one week later. The study was limited to one course "
    "and does not establish whether the effect is the same for all age groups or languages."
)

exercise_2_result = research_summary_pipeline(abstract_text)



============ Step 1 — Structured extraction ============
{
  "research_question": "Does spaced practice improve vocabulary retention among adult language learners?",
  "method": "Sixty adult language learners were randomly assigned to either daily short review sessions or one longer weekly session over four weeks, followed by a vocabulary recall test one week later.",
  "key_finding": "The spaced-practice group recalled more words on the test one week later, but the study was limited to one course and does not establish whether the effect is the same for all age groups or languages."
}

============ Step 2 — Layperson summary ============
**Short‑term study on spaced learning**

Researchers asked whether “spaced practice” – reviewing new words at shorter, regular intervals – helps adults keep vocabulary longer than a single, longer review. In the experiment, 60 adult learners were split into two groups. One group studied the words every day for a few minutes, while the other studied a

---
# Exercise 3  Interview Transcript to Candidate Scorecard Pipeline

**Inputs:** `transcript_text`, `skills_to_assess`

**Step 1:** Extract question-and-answer pairs using `PromptTemplate` + `PydanticOutputParser`.  
**Step 2:** Evaluate each requested skill using `ChatPromptTemplate` + `PydanticOutputParser`, recording evidence or a gap.  
**Step 3:** Create a structured scorecard and overall recommendation from the evaluation only.

This is a demonstration of structured evidence review, not a substitute for a fair, human-led hiring decision. A skill should be marked as not demonstrated when the transcript does not contain clear evidence.


In [46]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="Interview question or topic")
    answer: str = Field(description="Candidate answer, summarized faithfully")

class InterviewAnswers(BaseModel):
    answers: List[InterviewAnswer] = Field(description="Question and answer pairs extracted from the transcript")

class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being assessed")
    demonstrated: bool = Field(description="True only when the transcript contains clear evidence")
    evidence_or_gap: str = Field(description="Specific evidence from the answer or what evidence is missing")

class SkillEvaluations(BaseModel):
    evaluations: List[SkillEvaluation] = Field(description="One evaluation for each requested skill")

class ScorecardRow(BaseModel):
    skill: str = Field(description="Skill assessed")
    result: str = Field(description="Demonstrated or Not clearly demonstrated")
    evidence_or_gap: str = Field(description="Evidence or gap supporting the result")

class CandidateScorecard(BaseModel):
    scorecard: List[ScorecardRow] = Field(description="Structured hiring scorecard rows")
    overall_recommendation: str = Field(description="Evidence-based next step, not a definitive hiring decision")

answers_parser = PydanticOutputParser(pydantic_object=InterviewAnswers)
skills_parser = PydanticOutputParser(pydantic_object=SkillEvaluations)
scorecard_parser = PydanticOutputParser(pydantic_object=CandidateScorecard)

answers_prompt = PromptTemplate(
    template=(
        "Extract the interview questions/topics and the candidate's answers from this transcript. "
        "Keep the meaning accurate and do not infer skills that are not stated.\n\n"
        "Transcript:\n{transcript_text}\n\n{format_instructions}"
    ),
    input_variables=["transcript_text"],
    partial_variables={"format_instructions": answers_parser.get_format_instructions()},
)

skills_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical interviewer evaluating skill evidence. Assess only the supplied answers. Mark demonstrated=true only when there is clear evidence. If evidence is weak or absent, mark false and explain the gap. Return one evaluation per requested skill."),
    ("human", "Skills to assess: {skills_to_assess}\n\nExtracted interview answers:\n{answers_json}\n\n{format_instructions}")
]).partial(format_instructions=skills_parser.get_format_instructions())

scorecard_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a hiring panel coordinator. Convert the supplied skill evaluations into a structured scorecard. Keep the recommendation cautious and evidence-based; do not make a final hiring decision."),
    ("human", "Skill evaluations:\n{evaluations_json}\n\n{format_instructions}")
]).partial(format_instructions=scorecard_parser.get_format_instructions())

interview_step_1 = answers_prompt | llm | answers_parser
interview_step_2 = skills_prompt | llm | skills_parser
interview_step_3 = scorecard_prompt | llm | scorecard_parser

def candidate_scorecard_pipeline(transcript_text, skills_to_assess):
    answers = interview_step_1.invoke({"transcript_text": transcript_text})
    evaluations = interview_step_2.invoke({
        "skills_to_assess": ", ".join(skills_to_assess),
        "answers_json": answers.model_dump_json(indent=2),
    })
    scorecard = interview_step_3.invoke({"evaluations_json": evaluations.model_dump_json(indent=2)})
    print_section("Step 1 — Extracted answers", answers)
    print_section("Step 2 — Skill evidence evaluation", evaluations)
    print_section("Step 3 — Final scorecard", scorecard)
    return {"answers": answers, "evaluations": evaluations, "scorecard": scorecard}


In [47]:
transcript_text = (
    "Interviewer: How did you use Java in a project?\n"
    "Candidate: I built REST endpoints in Spring Boot and used service classes to keep business logic separate.\n"
    "Interviewer: How did you test the API?\n"
    "Candidate: I used Postman to check successful requests and invalid input, but I have not written automated integration tests yet.\n"
    "Interviewer: Tell me about SQL performance.\n"
    "Candidate: I know basic joins and indexes from coursework, but I have not tuned a production database."
)
skills_to_assess = ["Java and Spring Boot", "API testing", "SQL performance tuning"]

exercise_3_result = candidate_scorecard_pipeline(transcript_text, skills_to_assess)



============ Step 1 — Extracted answers ============
{
  "answers": [
    {
      "question": "How did you use Java in a project?",
      "answer": "I built REST endpoints in Spring Boot and used service classes to keep business logic separate."
    },
    {
      "question": "How did you test the API?",
      "answer": "I used Postman to check successful requests and invalid input, but I have not written automated integration tests yet."
    },
    {
      "question": "Tell me about SQL performance.",
      "answer": "I know basic joins and indexes from coursework, but I have not tuned a production database."
    }
  ]
}

============ Step 2 — Skill evidence evaluation ============
{
  "evaluations": [
    {
      "skill": "Java and Spring Boot",
      "demonstrated": true,
      "evidence_or_gap": "Built REST endpoints in Spring Boot and used service classes to separate business logic, showing practical use of Java and Spring Boot."
    },
    {
      "skill": "API testing",
      "

---
# Exercise 4 - Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline

**Input:** `event_description_text`

**Step 1:** Extract event details with `PromptTemplate` + `PydanticOutputParser`.  
**Step 2:** Use the extracted location and date to query the Open-Meteo API. The API response is normalized into a Pydantic model with `PydanticOutputParser`; weather values come from the API, not from an LLM guess.  
**Step 3:** Generate a recommendation with `ChatPromptTemplate` + `StrOutputParser`.

**Note:** Open-Meteo is used for this demonstration and does not require an API key. Forecast availability depends on the selected date and location. For dates outside its forecast range, the API may not return usable data.


In [48]:
class EventDetails(BaseModel):
    event_type: str = Field(description="Type of event")
    event_date: str = Field(description="Event date in YYYY-MM-DD format; use 'Not specified' if absent")
    location: str = Field(description="City or location name")
    indoor_outdoor_preference: str = Field(description="Indoor, Outdoor, or Flexible")


class WeatherDetails(BaseModel):
    temperature: str = Field(description="Temperature in Celsius, or unavailable")
    precipitation_chance: str = Field(description="Precipitation probability percentage, or unavailable")
    conditions: str = Field(description="Weather condition description or unavailable")


event_parser = PydanticOutputParser(pydantic_object=EventDetails)
weather_parser = PydanticOutputParser(pydantic_object=WeatherDetails)

event_prompt = PromptTemplate(
    template=(
        "Extract event details from the description. Use a date in YYYY-MM-DD format when a date is provided. "
        "Do not guess a missing date or location; write 'Not specified'. Indoor/outdoor preference must be "
        "Indoor, Outdoor, or Flexible.\n\n"
        "Description:\n{event_description_text}\n\n"
        "{format_instructions}"
    ),
    input_variables=["event_description_text"],
    partial_variables={
        "format_instructions": event_parser.get_format_instructions()
    },
)

# Create the weather query using the extracted event details.
weather_query_prompt = PromptTemplate.from_template(
    "Look up weather for location={location} on date={event_date}. "
    "Use the returned values from the weather service only."
)

event_recommendation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an event planning assistant. Recommend whether to proceed "
        "as planned, reschedule, or move indoors. Base your recommendation "
        "only on the supplied event details and weather data. Consider high "
        "temperatures, precipitation chance, weather conditions, outdoor "
        "exposure, and the availability of an indoor alternative. Treat "
        "temperatures around 33 degrees Celsius as hot, not mild. Mention "
        "practical precautions such as shade, drinking water, and breaks "
        "when relevant. If weather data is unavailable, incomplete, or "
        "uncertain, clearly state the limitation and do not invent a forecast. "
        "Do not claim that an event is completely safe based on weather data alone."
    ),
    (
        "human",
        "Event details:\n{event_json}\n\n"
        "Weather data:\n{weather_json}\n\n"
        "Write a concise recommendation to the organizer. Explain the reason "
        "for your recommendation and mention relevant precautions."
    ),
])

event_step_1 = event_prompt | llm | event_parser
event_step_3 = event_recommendation_prompt | llm | str_parser


def fetch_open_meteo_weather(location, event_date):
    # Find the coordinates for the requested location.
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": location,
            "count": 1,
            "language": "en",
            "format": "json",
        },
        timeout=20,
    )
    geo_response.raise_for_status()

    geo_results = geo_response.json().get("results", [])

    if not geo_results:
        raise ValueError(f"Could not find a location for: {location}")

    place = geo_results[0]

    # Format the API query using the extracted location and date.
    query = weather_query_prompt.invoke({
        "location": f"{place['name']}, {place.get('country', '')}",
        "event_date": event_date,
    })
    print("Weather query:", query.to_string())

    # Request daily weather data from Open-Meteo.
    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "daily": (
                "temperature_2m_max,"
                "precipitation_probability_max,"
                "weather_code"
            ),
            "timezone": "auto",
            "start_date": event_date,
            "end_date": event_date,
        },
        timeout=20,
    )
    forecast_response.raise_for_status()

    daily = forecast_response.json().get("daily", {})
    dates = daily.get("time", [])

    if not dates:
        return WeatherDetails(
            temperature="Unavailable for this date",
            precipitation_chance="Unavailable for this date",
            conditions="No forecast returned by the weather service",
        )

    temp = daily.get("temperature_2m_max", [None])[0]
    precip = daily.get("precipitation_probability_max", [None])[0]
    code = daily.get("weather_code", [None])[0]

    condition_map = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",
        80: "Rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        95: "Thunderstorm",
        96: "Thunderstorm with hail",
        99: "Thunderstorm with heavy hail",
    }

    # Keep the actual API values; do not invent missing weather data.
    raw_weather = {
        "temperature": (
            f"{temp} °C" if temp is not None else "Unavailable"
        ),
        "precipitation_chance": (
            f"{precip}%" if precip is not None else "Unavailable"
        ),
        "conditions": condition_map.get(
            code,
            f"Weather code {code}" if code is not None else "Unavailable",
        ),
    }

    # Parse the normalized weather data into the required Pydantic model.
    return weather_parser.parse(json.dumps(raw_weather))


def event_weather_pipeline(event_description_text):
    # Step 1: Extract the event details.
    event = event_step_1.invoke({
        "event_description_text": event_description_text
    })

    if (
        event.event_date == "Not specified"
        or event.location == "Not specified"
    ):
        raise ValueError(
            "Please provide a specific event date and location before requesting weather."
        )

    # Step 2: Retrieve and structure the actual weather data.
    weather = fetch_open_meteo_weather(
        event.location,
        event.event_date,
    )

    # Step 3: Generate the recommendation from structured data.
    recommendation = event_step_3.invoke({
        "event_json": event.model_dump_json(indent=2),
        "weather_json": weather.model_dump_json(indent=2),
    })

    print_section("Step 1 — Event details", event)
    print_section("Step 2 — Weather data", weather)
    print_section("Step 3 — Organizer recommendation", recommendation)

    return {
        "event": event,
        "weather": weather,
        "recommendation": recommendation,
    }

In [49]:
event_description_text = (
    "We are planning an outdoor community book fair in Pune on 2026-10-25. "
    "The stalls and children's reading area will be outside, but we can move the main activities indoors if needed."
)

# Forecast availability varies by date. If this date is outside the API's forecast window,
# the notebook will report that the weather data is unavailable rather than inventing it.
try:
    exercise_4_result = event_weather_pipeline(event_description_text)
except (requests.RequestException, ValueError, KeyError) as error:
    print("Weather pipeline could not complete:", error)
    print("Check the event date/location or try a date supported by the forecast API.")


Weather query: Look up weather for location=Pune, India on date=2026-10-25. Use the returned values from the weather service only.

============ Step 1 — Event details ============
{
  "event_type": "community book fair",
  "event_date": "2026-10-25",
  "location": "Pune",
  "indoor_outdoor_preference": "Flexible"
}

============ Step 2 — Weather data ============
{
  "temperature": "33.1 °C",
  "precipitation_chance": "7%",
  "conditions": "Overcast"
}

============ Step 3 — Organizer recommendation ============
**Recommendation:** Proceed outdoors with precautions (no reschedule or indoor move required).

**Reasoning:**  
- **Temperature:** 33.1 °C is classified as hot, so the event will be warm for attendees.  
- **Precipitation:** Only a 7 % chance of rain, so flooding or sudden showers are unlikely.  
- **Conditions:** Overcast skies reduce direct sun exposure, but the high temperature still poses heat‑related risks.  
- **Indoor alternative:** Since the event is “flexible,” an in

---
# Exercise 5 - Support Chat Log to Escalation Report Pipeline

**Input:** `chat_log_text`

**Step 1:** Normalize the chat into a structured issue summary.  
**Step 2:** Check the summary for missing escalation information and readiness.  
**Step 3:** Produce a prioritized report using the structured summary and gap check, not the raw chat log.


In [50]:
class SupportSummary(BaseModel):
    issue: str = Field(description="Main problem reported by the customer")
    customer_tone: str = Field(description="Customer tone, based on the text")
    resolution_attempted: str = Field(description="Troubleshooting or resolution already attempted; say 'Not stated' if absent")

class EscalationReadiness(BaseModel):
    missing_fields: List[str] = Field(description="Important missing details such as account ID, error code, or timestamps")
    is_ready_to_escalate: bool = Field(description="True only if enough information exists to escalate usefully")

support_parser = PydanticOutputParser(pydantic_object=SupportSummary)
readiness_parser = PydanticOutputParser(pydantic_object=EscalationReadiness)

support_extract_prompt = PromptTemplate(
    template=(
        "Normalize this support chat into the issue, customer tone, and resolution attempted. "
        "Do not invent missing facts.\n\nChat log:\n{chat_log_text}\n\n{format_instructions}"
    ),
    input_variables=["chat_log_text"],
    partial_variables={"format_instructions": support_parser.get_format_instructions()},
)

readiness_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an escalation-readiness reviewer. Check whether the structured summary includes enough detail for a useful handoff. Consider account ID, error code, affected service, and relevant time when applicable. Do not invent missing values. List missing information explicitly."),
    ("human", "Structured support summary:\n{summary_json}\n\n{format_instructions}")
]).partial(format_instructions=readiness_parser.get_format_instructions())

escalation_report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a support operations lead. Write a concise escalation report "
        "using only the structured support summary and readiness review. "
        "Include priority, reason, issue, resolution attempted, missing "
        "information, and recommended next action. Use this priority scale: "
        "P1 means a critical widespread outage or similarly severe impact; "
        "P2 means a major service issue with significant impact; "
        "P3 means a standard issue with limited or unclear impact; "
        "P4 means a minor issue or request. Do not assign P1 without evidence "
        "of critical impact. If the available information is insufficient, "
        "state that priority is provisional and explain why. Never invent "
        "account details, error codes, or timestamps."
    ),
    (
        "human",
        "Support summary:\n{summary_json}\n\n"
        "Readiness review:\n{readiness_json}\n\n"
        "Prepare the escalation report."
    ),
])


support_step_1 = support_extract_prompt | llm | support_parser
support_step_2 = readiness_prompt | llm | readiness_parser
support_step_3 = escalation_report_prompt | llm | str_parser

def escalation_pipeline(chat_log_text):
    summary = support_step_1.invoke({"chat_log_text": chat_log_text})
    readiness = support_step_2.invoke({"summary_json": summary.model_dump_json(indent=2)})
    report = support_step_3.invoke({
        "summary_json": summary.model_dump_json(indent=2),
        "readiness_json": readiness.model_dump_json(indent=2),
    })
    print_section("Step 1 — Structured support summary", summary)
    print_section("Step 2 — Escalation readiness", readiness)
    print_section("Step 3 — Escalation report", report)
    return {"summary": summary, "readiness": readiness, "report": report}


In [51]:
chat_log_text = (
    "Customer: The checkout keeps failing and I have tried again three times.\n"
    "Agent: Please clear your browser cache and try once more.\n"
    "Customer: I did that, but it still says 'payment could not be processed'. "
    "I need to place the order today.\n"
    "Agent: I will pass this to the technical team.\n"
    "Note: No account ID, transaction reference, or exact error code was recorded."
)

exercise_5_result = escalation_pipeline(chat_log_text)



============ Step 1 — Structured support summary ============
{
  "issue": "Checkout/payment failure during order placement",
  "customer_tone": "Frustrated and urgent",
  "resolution_attempted": "Cleared browser cache; escalated to technical team"
}

============ Step 2 — Escalation readiness ============
{
  "missing_fields": [
    "account ID",
    "error code",
    "affected service",
    "timestamp"
  ],
  "is_ready_to_escalate": false
}

============ Step 3 — Escalation report ============
**Escalation Report**

- **Priority:** *Provisional P3* – The issue is a checkout/payment failure, but lack of critical impact evidence (no outage scope, error code, or timestamp).  
- **Reason for Priority:** Standard issue with limited impact until further details confirm broader service disruption.  
- **Issue:** Checkout/payment failure during order placement.  
- **Resolution Attempted:** Customer cleared browser cache; case escalated to the technical team.  
- **Missing Information:**  


---
# Exercise 6 - Workout Goal to Weekly Plan Pipeline

**Inputs:** `fitness_goal_text`, `days_per_week`

**Step 1:** Extract goal details using `PromptTemplate` + `PydanticOutputParser`.  
**Step 2:** Generate a structured day-by-day plan using `ChatPromptTemplate` + `PydanticOutputParser`, based only on the extracted details.

The sample is general fitness information, not medical advice. Anyone with an injury or medical concern should seek advice from a qualified professional before starting a new exercise plan.


In [52]:
class FitnessGoalDetails(BaseModel):
    goal_type: str = Field(description="Main fitness goal")
    fitness_level: str = Field(description="Current fitness level as described")
    days_available: int = Field(description="Number of available workout days per week")
    equipment_access: str = Field(
        description="Equipment access; say 'Not specified' if absent"
    )


class WorkoutDay(BaseModel):
    day: str = Field(description="Day name or workout day number")
    focus: str = Field(description="Workout focus")
    activities: List[str] = Field(
        description="Exercises or activities for this day"
    )
    duration_minutes: int = Field(
        ge=0,
        le=180,
        description="Approximate workout duration in minutes; use 0 for rest"
    )
    notes: str = Field(description="Rest, warm-up, or safety note")


class WeeklyWorkoutPlan(BaseModel):
    plan: List[WorkoutDay] = Field(
        description="Day-by-day plan entries; include rest days as needed"
    )


fitness_parser = PydanticOutputParser(
    pydantic_object=FitnessGoalDetails
)

workout_parser = PydanticOutputParser(
    pydantic_object=WeeklyWorkoutPlan
)


fitness_extract_prompt = PromptTemplate(
    template=(
        "Extract the person's fitness goal, current level, available days, "
        "and equipment access. Use the supplied days_per_week value for "
        "days_available. Do not infer medical conditions. If equipment "
        "access is not mentioned, write 'Not specified'.\n\n"
        "Goal description:\n{fitness_goal_text}\n"
        "Days available per week: {days_per_week}\n\n"
        "{format_instructions}"
    ),
    input_variables=["fitness_goal_text", "days_per_week"],
    partial_variables={
        "format_instructions": fitness_parser.get_format_instructions()
    },
)


workout_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a certified personal trainer. Create a realistic, "
        "beginner-appropriate weekly workout plan using only the supplied "
        "extracted details. Schedule workouts on the available days and "
        "include rest days as appropriate. Include warm-up and cool-down "
        "guidance. Do not assume equipment that is not available. "
        "For every plan entry, duration_minutes must be an integer from "
        "0 to 180. Use 0 for a rest day. Return valid structured output "
        "following the supplied format instructions."
    ),
    (
        "human",
        "Extracted goal details:\n{goal_details_json}\n\n"
        "{format_instructions}"
    ),
]).partial(
    format_instructions=workout_parser.get_format_instructions()
)


fitness_step_1 = fitness_extract_prompt | llm | fitness_parser
fitness_step_2 = workout_prompt | llm | workout_parser


def workout_pipeline(fitness_goal_text, days_per_week):
    if not isinstance(days_per_week, int) or not 1 <= days_per_week <= 7:
        raise ValueError("days_per_week must be an integer from 1 to 7.")

    # Step 1: Extract the fitness goal details.
    goal = fitness_step_1.invoke({
        "fitness_goal_text": fitness_goal_text,
        "days_per_week": days_per_week,
    })

    # Step 2: Generate the structured weekly workout plan.
    plan = fitness_step_2.invoke({
        "goal_details_json": goal.model_dump_json(indent=2)
    })

    print_section("Step 1 — Extracted fitness goal", goal)
    print_section("Step 2 — Weekly workout plan", plan)

    return {
        "goal": goal,
        "plan": plan,
    }

In [53]:
fitness_goal_text = (
    "I am new to regular exercise and want to improve general fitness and stamina. "
    "I can exercise on weekdays three times each week. I do not have gym equipment, "
    "so I prefer walking and bodyweight exercises."
)

exercise_6_result = workout_pipeline(fitness_goal_text, days_per_week=3)



============ Step 1 — Extracted fitness goal ============
{
  "goal_type": "Improve general fitness and stamina",
  "fitness_level": "New to regular exercise",
  "days_available": 3,
  "equipment_access": "No gym equipment"
}

============ Step 2 — Weekly workout plan ============
{
  "plan": [
    {
      "day": "Day 1",
      "focus": "Cardio & Core",
      "activities": [
        "Jumping jacks",
        "High knees",
        "Bodyweight squats",
        "Modified push‑ups",
        "Plank",
        "Mountain climbers"
      ],
      "duration_minutes": 45,
      "notes": "Warm‑up: 5 min brisk marching in place and dynamic leg swings. Cool‑down: 5 min static stretches for legs, chest, and shoulders."
    },
    {
      "day": "Day 2",
      "focus": "Rest",
      "activities": [],
      "duration_minutes": 0,
      "notes": "Rest and recovery. Light walking or gentle stretching optional."
    },
    {
      "day": "Day 3",
      "focus": "Full Body Strength",
      "activities": [


In [54]:
plan = exercise_6_result["plan"]

print("Total plan entries:", len(plan.plan))

for item in plan.plan:
    print(
        f"{item.day}: {item.focus} | "
        f"{item.duration_minutes} minutes"
    )

Total plan entries: 5
Day 1: Cardio & Core | 45 minutes
Day 2: Rest | 0 minutes
Day 3: Full Body Strength | 45 minutes
Day 4: Rest | 0 minutes
Day 5: Endurance & Mobility | 45 minutes
